# 02 — Feature Engineering and Risk Labels

This notebook transforms the cleaned USD/RWF daily data into a machine-learning-ready dataset.

Features include:
- daily return
- 7-day and 14-day returns
- moving averages
- volatility
- momentum
- bid/ask spread
- depreciation-day counts

Labels:
- 7-day depreciation risk: Low / Medium / High
- 14-day depreciation risk: Low / Medium / High

In [3]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent
PROCESSED_DIR = ROOT / 'data' / 'processed'

daily = pd.read_csv(PROCESSED_DIR / 'exchange_rates_daily_calendar_4year.csv', parse_dates=['date'])
daily = daily.sort_values(['currency', 'date']).reset_index(drop=True)
daily.head()

,date,currency,buying_rate,average_rate,selling_rate,mid_rate,is_official_observation
0,2022-06-26,USD,1013.97,1024.10,1034.24,1024.10,0
1,2022-06-27,USD,1013.97,1024.10,1034.24,1024.10,1
2,2022-06-28,USD,1013.77,1023.90,1034.04,1023.90,1
3,2022-06-29,USD,1014.06,1024.20,1034.34,1024.20,1
4,2022-06-30,USD,1014.34,1024.48,1034.62,1024.48,1


In [8]:
def add_features(group):
    g = group.sort_values('date').copy()
    g['daily_return'] = g['mid_rate'].pct_change()
    g['return_7d'] = g['mid_rate'].pct_change(7)
    g['return_14d'] = g['mid_rate'].pct_change(14)
    g['ma_7'] = g['mid_rate'].rolling(7).mean()
    g['ma_14'] = g['mid_rate'].rolling(14).mean()
    g['ma_30'] = g['mid_rate'].rolling(30).mean()
    g['ma_gap'] = (g['ma_7'] - g['ma_30']) / g['ma_30']
    g['volatility_7d'] = g['daily_return'].rolling(7).std()
    g['volatility_14d'] = g['daily_return'].rolling(14).std()
    g['volatility_30d'] = g['daily_return'].rolling(30).std()
    g['momentum_7d'] = g['return_7d']
    g['momentum_14d'] = g['return_14d']
    g['spread'] = g['selling_rate'] - g['buying_rate']
    g['spread_pct'] = g['spread'] / g['mid_rate']
    g['depreciated_today'] = (g['daily_return'] > 0).astype(int)
    g['depreciation_days_7d'] = g['depreciated_today'].rolling(7).sum()
    g['depreciation_days_14d'] = g['depreciated_today'].rolling(14).sum()
    g['future_7d_change'] = g['mid_rate'].shift(-7) / g['mid_rate'] - 1
    g['future_14d_change'] = g['mid_rate'].shift(-14) / g['mid_rate'] - 1
    return g.drop(columns=['depreciated_today'])

features = daily.groupby('currency', group_keys=False).apply(add_features).reset_index(drop=True)
features['currency'] = 'USD'
features.head(35).tail()

,date,buying_rate,average_rate,selling_rate,mid_rate,is_official_observation,daily_return,return_7d,return_14d,ma_7,...,volatility_30d,momentum_7d,momentum_14d,spread,spread_pct,depreciation_days_7d,depreciation_days_14d,future_7d_change,future_14d_change,currency
30,2022-07-26,1017.79,1027.97,1038.15,1027.97,1,0.000282,0.001452,0.002604,1027.360000,...,0.000141,0.001452,0.002604,20.36,0.019806,5.0,10.0,0.001274,0.002383,USD
31,2022-07-27,1018.10,1028.28,1038.46,1028.28,1,0.000302,0.001480,0.002740,1027.577143,...,0.000143,0.001480,0.002740,20.36,0.019800,5.0,10.0,0.001255,0.002392,USD
32,2022-07-28,1018.40,1028.58,1038.76,1028.58,1,0.000292,0.001480,0.002788,1027.794286,...,0.000131,0.001480,0.002788,20.36,0.019794,5.0,10.0,0.001274,0.002382,USD
33,2022-07-29,1018.64,1028.82,1039.01,1028.82,1,0.000233,0.001431,0.002797,1028.004286,...,0.000129,0.001431,0.002797,20.37,0.019799,5.0,10.0,0.001040,0.002440,USD
34,2022-07-30,1018.64,1028.82,1039.01,1028.82,0,0.000000,0.001431,0.002797,1028.214286,...,0.000130,0.001431,0.002797,20.37,0.019799,5.0,10.0,0.001040,0.002440,USD


## Create Low / Medium / High labels

Because USD/RWF movements are usually not extremely large daily, quantile-based labels are used. This adapts the labels to the real distribution of Rwanda's exchange-rate movement.

- bottom 50% = Low
- 50% to 80% = Medium
- top 20% = High

In [9]:
def make_risk_label(series):
    q50 = series.quantile(0.50)
    q80 = series.quantile(0.80)
    labels = pd.Series(index=series.index, dtype='object')
    labels[series <= q50] = 'Low'
    labels[(series > q50) & (series <= q80)] = 'Medium'
    labels[series > q80] = 'High'
    return labels, q50, q80

features['risk_label_7d'], q50_7, q80_7 = make_risk_label(features['future_7d_change'])
features['risk_label_14d'], q50_14, q80_14 = make_risk_label(features['future_14d_change'])

print('7-day thresholds:', q50_7, q80_7)
print('14-day thresholds:', q50_14, q80_14)
features[['date', 'currency', 'mid_rate', 'future_7d_change', 'risk_label_7d', 'future_14d_change', 'risk_label_14d']].tail(20)

7-day thresholds: 0.0015660065755532804 0.0026750329145906362
14-day thresholds: 0.0031488225162865646 0.005390612010726992


,date,currency,mid_rate,future_7d_change,risk_label_7d,future_14d_change,risk_label_14d
1441,2026-06-06,USD,1464.182,0.000296,Low,0.000606,Low
1442,2026-06-07,USD,1464.182,0.000296,Low,0.000606,Low
1443,2026-06-08,USD,1464.279,0.000291,Low,0.000605,Low
1444,2026-06-09,USD,1464.360,0.000300,Low,0.000615,Low
1445,2026-06-10,USD,1464.450,0.000300,Low,0.000615,Low
1446,2026-06-11,USD,1464.545,0.000300,Low,0.000615,Low
1447,2026-06-12,USD,1464.615,0.000311,Low,NaN,NaN
1448,2026-06-13,USD,1464.615,0.000311,Low,NaN,NaN
1449,2026-06-14,USD,1464.615,0.000311,Low,NaN,NaN
1450,2026-06-15,USD,1464.705,0.000314,Low,NaN,NaN


In [10]:
features.to_csv(PROCESSED_DIR / 'exchange_rates_features_and_labels_4year.csv', index=False)

FEATURE_COLUMNS = [
    'mid_rate', 'daily_return', 'return_7d', 'return_14d', 'ma_7', 'ma_14', 'ma_30',
    'ma_gap', 'volatility_7d', 'volatility_14d', 'volatility_30d', 'momentum_7d',
    'momentum_14d', 'spread', 'spread_pct', 'depreciation_days_7d', 'depreciation_days_14d'
]

model_7d = features.dropna(subset=FEATURE_COLUMNS + ['future_7d_change', 'risk_label_7d']).copy()
model_14d = features.dropna(subset=FEATURE_COLUMNS + ['future_14d_change', 'risk_label_14d']).copy()

model_7d[['date','currency'] + FEATURE_COLUMNS + ['future_7d_change','risk_label_7d']].to_csv(
    PROCESSED_DIR / 'exchange_rates_model_ready_7d_4year.csv', index=False
)
model_14d[['date','currency'] + FEATURE_COLUMNS + ['future_14d_change','risk_label_14d']].to_csv(
    PROCESSED_DIR / 'exchange_rates_model_ready_14d_4year.csv', index=False
)

print('7-day model-ready:', model_7d.shape)
print('14-day model-ready:', model_14d.shape)

7-day model-ready: (1424, 27)
14-day model-ready: (1417, 27)


In [11]:
print('7-day label distribution')
print(model_7d['risk_label_7d'].value_counts(normalize=True).round(3))
print('14-day label distribution')
print(model_14d['risk_label_14d'].value_counts(normalize=True).round(3))

7-day label distribution
risk_label_7d
Low       0.489
Medium    0.306
High      0.204
Name: proportion, dtype: float64
14-day label distribution
risk_label_14d
Low       0.490
Medium    0.306
High      0.205
Name: proportion, dtype: float64
